# Chemostat Operating Mode Test Notebook

This notebook tests the **CHEMOSTAT** operating mode implementation.

**Test Coverage:**
- Simulation setup and execution
- Single batch run with initial conditions
- Multi-batch run with auto-extracted initial conditions
- Data integrity validation
- Performance metrics
- Chemostat-specific validations (steady-state, constant volume, dilution effects)

## 1. Setup and Imports

In [1]:
import sys
import numpy as np
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from animakinetica.core.simulation import CellCultureSimulation
from animakinetica.core.parameters import SimulationParameters
from animakinetica.models.kinetics import Kinetics
from animakinetica.models.odesystems import OperatingMode
from animakinetica.models.gas_controllers import GasController
from animakinetica.models.feed_strategies import FeedStrategy
from animakinetica.models.stoichiometry import StoichiometryMode


print("✓ Imports successful")

✓ Imports successful


## 2. Load Parameters and Initialize Simulation

In [2]:
# Load base parameters
params = SimulationParameters.load_parameters(["Cell_GFI", "anab_GFI", "cata_GFI", "fed_batch", "chemostat"])
params.update("excess_kinetics_batch")
    
# Initialize simulation with CHEMOSTAT mode
sim = CellCultureSimulation(
    parameters=params,
    kinetics=Kinetics.EXCESS,
    stoichiometry_mode=StoichiometryMode.DYNAMIC, 
    feed_strategy=FeedStrategy.EXPONENTIAL,
    operating_mode=OperatingMode.CHEMOSTAT,
    gas_controller=GasController.CASCADE_N_FG_XO2,
    performance_mode=False,
    solver_method="BDF"

)

print(f"✓ Simulation initialized")
print(f"  Operating Mode: {sim.operating_mode}")
print(f"  Kinetics Model: {sim.kinetics}")
print(f"  Gas Controller: {sim.gas_controller}")
print(f"  Dilution Rate: {float(params.chemostat.D):.4f} 1/h")

Loading parameters from 6 file(s) sequentially...
  [1/6] Base: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\parameter_sets\base_parameters.yaml
  [2/6] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\parameter_sets\cell_properties\Cell_GFI.yaml
  [3/6] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\parameter_sets\stoichiometry\anab_GFI.yaml
  [4/6] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\parameter_sets\stoichiometry\cata_GFI.yaml
  [5/6] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\parameter_sets\operating_modes\fed_batch.yaml
  [6/6] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\pa

## 4. Multi-Batch Run

In [3]:
# Run 5 repeated batches
num_batches = 5
multi_result = sim.run_simulation(number_of_batches=num_batches)

print(f"✓ {num_batches} batches completed")
print(f"\nSummary Statistics:")
print(f"  Average batch duration: {multi_result.projected_totals_stats['total_simulation_time_h']/num_batches:.2f} h")
print(f"  STY: {multi_result.summary_stats['STY_kg_m3_day']:.2f} kg/m3/day")

print(f"\nBatch-by-Batch Results:")
for i, batch in enumerate(sim.data.batches, 1):
    print(f"  Batch {i}: {batch.t_end:.2f} h, Xv={batch.Xv_final:.2e} cells/L, V={batch.V_final:.2f} L")

run_simulation execution time: 1782.935 ms
✓ 5 batches completed

Summary Statistics:
  Average batch duration: 979.20 h
  STY: 0.25 kg/m3/day

Batch-by-Batch Results:
  Batch 1: 960.00 h, Xv=3.95e+09 cells/L, V=1600.00 L
  Batch 2: 960.00 h, Xv=3.95e+09 cells/L, V=1600.00 L
  Batch 3: 960.00 h, Xv=3.95e+09 cells/L, V=1600.00 L
  Batch 4: 960.00 h, Xv=3.95e+09 cells/L, V=1600.00 L
  Batch 5: 960.00 h, Xv=3.95e+09 cells/L, V=1600.00 L


## 5. Data Integrity Validation

In [4]:
# Test batch 1 data integrity
batch_1 = sim.data.batches[0]

print("Data Integrity Checks for Batch 1:")
print("\n1. State Variables:")
# Explicit index mapping: [Xv, C_Glc, C_Gln, C_Lac, C_Amm, V, C_O2, C_CT, C_H, C_Na, C_AA]
state_var_indices = {
    'Xv': 0, 'C_Glc': 1, 'C_Gln': 2, 'C_Lac': 3, 'C_Amm': 4,
    'V': 5, 'C_O2': 6, 'C_CT': 7, 'C_H': 8, 'C_Na': 9, 'C_AA': 10
}
for var, idx in state_var_indices.items():
    data = np.asarray(batch_1.y_points[idx], dtype=float)
    print(f"  {var} (idx {idx}): {len(data)} points, range [{data.min():.2e}, {data.max():.2e}]")
    assert len(data) > 0, f"{var} data is empty"
    assert not np.any(np.isnan(data)), f"{var} contains NaN values"

print("\n2. Kinetic Rates:")
kinetic_rates = ['mu', 'q_Glc', 'q_Gln', 'q_Lac', 'q_Amm', 'q_O2', 'q_CO2', 'q_AA']
for rate in kinetic_rates:
    rate_list = getattr(batch_1, f'{rate}_list')
    print(f"  {rate}: {len(rate_list)} points, range [{rate_list.min():.2e}, {rate_list.max():.2e}]")
    assert len(rate_list) > 0, f"{rate}_list is empty"
    assert not np.any(np.isnan(rate_list)), f"{rate}_list contains NaN values"

print("\n3. Nutrient Factors:")
nf_vars = ['NF_glucose', 'NF_glutamine', 'NF_O2', 'NF']
for nf in nf_vars:
    nf_list = getattr(batch_1, f'{nf}_list')
    print(f"  {nf}: {len(nf_list)} points, range [{nf_list.min():.2e}, {nf_list.max():.2e}]")
    assert len(nf_list) > 0, f"{nf}_list is empty"
    assert np.all((nf_list >= 0) & (nf_list <= 1)), f"{nf}_list contains values outside [0,1]"

print("\n4. Inhibition Factors:")
if_vars = ['IF_O2', 'IF_CO2', 'IF_lactate', 'IF_ammonia', 'IF_osmolality', 'IF']
for ifvar in if_vars:
    if_list = getattr(batch_1, f'{ifvar}_list')
    print(f"  {ifvar}: {len(if_list)} points, range [{if_list.min():.2e}, {if_list.max():.2e}]")
    assert len(if_list) > 0, f"{ifvar}_list is empty"
    assert np.all((if_list >= 0) & (if_list <= 1)), f"{ifvar}_list contains values outside [0,1]"

print("\n5. Gas Control Variables:")
gas_vars = ['N', 'Fg', 'y_O2', 'kLa', 'OTR', 'CTR', 'C_CO2']
for gv in gas_vars:
    gv_list = getattr(batch_1, f'{gv}_list')
    print(f"  {gv}: {len(gv_list)} points, range [{gv_list.min():.2e}, {gv_list.max():.2e}]")
    assert len(gv_list) > 0, f"{gv}_list is empty"
    assert not np.any(np.isnan(gv_list)), f"{gv}_list contains NaN values"

print("\n6. pH Control Variables:")
print(f"  D_base: {len(batch_1.D_base_list)} points, range [{batch_1.D_base_list.min():.2e}, {batch_1.D_base_list.max():.2e}]")
assert len(batch_1.D_base_list) > 0, "D_base_list is empty"

print("\n7. Dilution Rate (Chemostat-specific):")
D_list = batch_1.D_list
print(f"  D: {len(D_list)} points, range [{D_list.min():.4f}, {D_list.max():.4f}] 1/h")
assert len(D_list) > 0, "D_list is empty"
assert not np.any(np.isnan(D_list)), "D_list contains NaN values"

print("\n✓ All data integrity checks passed")


Data Integrity Checks for Batch 1:

1. State Variables:
  Xv (idx 0): 987 points, range [6.00e+08, 4.73e+09]
  C_Glc (idx 1): 987 points, range [5.00e+00, 1.50e+01]
  C_Gln (idx 2): 987 points, range [1.64e+00, 5.00e+00]
  C_Lac (idx 3): 987 points, range [0.00e+00, 1.75e+01]
  C_Amm (idx 4): 987 points, range [0.00e+00, 4.92e+00]
  V (idx 5): 987 points, range [4.80e+02, 1.60e+03]
  C_O2 (idx 6): 987 points, range [1.00e-02, 1.00e-02]
  C_CT (idx 7): 987 points, range [9.15e+00, 2.90e+01]
  C_H (idx 8): 987 points, range [6.31e-08, 6.31e-08]
  C_Na (idx 9): 987 points, range [0.00e+00, 0.00e+00]
  C_AA (idx 10): 987 points, range [2.92e+00, 8.02e+00]

2. Kinetic Rates:
  mu: 987 points, range [1.82e-02, 2.72e-02]
  q_Glc: 987 points, range [5.73e-11, 6.21e-11]
  q_Gln: 987 points, range [1.69e-11, 1.91e-11]
  q_Lac: 987 points, range [7.64e-11, 8.32e-11]
  q_Amm: 987 points, range [1.65e-11, 2.37e-11]
  q_O2: 987 points, range [7.37e-11, 1.10e-10]
  q_CO2: 987 points, range [9.06e-11,

## 6. Performance Metrics

In [5]:
print("Performance Metrics (5-Batches):")
print(f"\n Total Biomass: {multi_result.projected_totals_stats['total_DCM_out_kg']:.2f} kg")
print(f"  STY: {multi_result.summary_stats['STY_kg_m3_day']:.3f} kg/m3/day")
print(f"  Medium Efficiency: {multi_result.summary_stats['medium_use_efficiency_kg_m3']:.3f} kg/m3")
print(f"  Protein Use Efficiency: {multi_result.summary_stats['protein_use_efficiency_kg_kg']:.3f} kg/kg")
print(f"  Nutrient Use Efficiency: {multi_result.summary_stats['nutrient_use_efficiency_kg_kg']:.3f} kg/kg")
print(f"  Total Simulation Time: {multi_result.projected_totals_stats['total_simulation_time_h']:.2f} h")
print(f"  Number of Batches: {multi_result.projected_totals_stats['number_of_batches']}")


# Validate metrics are reasonable
assert multi_result.projected_totals_stats['total_DCM_out_kg'] > 0, "Total DCM should be positive"
assert multi_result.summary_stats['STY_kg_m3_day'] > 0, "STY should be positive"
assert 0 < multi_result.summary_stats['medium_use_efficiency_kg_m3'] < 1000, "Medium use efficiency should be between 0 and 100 kg/m3"
assert 0 < multi_result.summary_stats['protein_use_efficiency_kg_kg'] < 1, "Protein use efficiency should be between 0 and 1"
assert 0 < multi_result.summary_stats['nutrient_use_efficiency_kg_kg'] < 1, "Nutrient use efficiency should be between 0 and 1"
assert multi_result.projected_totals_stats['total_simulation_time_h'] > 0, "Total simulation time should be positive"
assert multi_result.projected_totals_stats['number_of_batches'] == num_batches, f"Number of batches should be {num_batches}"

print("\n✓ Performance metrics validated")

Performance Metrics (5-Batches):

 Total Biomass: 83.01 kg
  STY: 0.254 kg/m3/day
  Medium Efficiency: 0.596 kg/m3
  Protein Use Efficiency: 0.198 kg/kg
  Nutrient Use Efficiency: 0.109 kg/kg
  Total Simulation Time: 4896.00 h
  Number of Batches: 5

✓ Performance metrics validated


## 7. Chemostat-Specific Validations

In [6]:
print("Chemostat-Specific Validations:")

# 1. Steady-state convergence (check if state variables stabilize)
print("\n1. Steady-State Convergence Analysis (Batch 1):")
# Analyze last 20% of run for steady-state
n_points = len(batch_1.t_points)
ss_start_idx = int(0.8 * n_points)

state_indices = {'Xv': 0, 'C_Glc': 1, 'C_Gln': 2, 'C_Lac': 3, 'C_Amm': 4, 'C_AA': 10}
for var_name, idx in state_indices.items():
    data_full = batch_1.y_points[idx, :]
    data_ss = data_full[ss_start_idx:]
    mean_ss = data_ss.mean()
    std_ss = data_ss.std()
    cv_ss = (std_ss / mean_ss) * 100 if mean_ss > 0 else 0
    print(f"  {var_name} (idx {idx}): mean={mean_ss:.2e}, std={std_ss:.2e}, CV={cv_ss:.2f}%")
    if cv_ss < 5.0:
        print(f"    ✓ Reached steady-state")
    else:
        print(f"    → Still converging (CV > 5%)")

# 2. Growth rate vs dilution rate relationship
print("\n2. Growth Rate vs Dilution Rate:")
# In chemostat steady-state: mu = D (washout avoided when mu > D)
for i, batch in enumerate(sim.data.batches, 1):
    mu_mean = batch.mu_list[ss_start_idx:].mean()
    D_mean = batch.D_list[ss_start_idx:].mean()
    print(f"  Batch {i}: mu_mean={mu_mean:.4f}, D_mean={D_mean:.4f} 1/h")
    print(f"           mu/D ratio: {mu_mean/D_mean:.3f}")
    net_growth = mu_mean - D_mean
    print(f"           Net growth (mu-D): {net_growth:.4f} 1/h")

# 3. Substrate levels in steady-state
print("\n3. Steady-State Substrate Levels:")
C_Glc_feed = float(params.chemostat.C_Glc_feed)
C_Gln_feed = float(params.chemostat.C_Gln_feed)
C_AA_feed  = float(params.fed_batch.C_AA_feed)
for i, batch in enumerate(sim.data.batches, 1):
    C_Glc_ss = batch.y_points[1, ss_start_idx:].mean()
    C_Gln_ss = batch.y_points[2, ss_start_idx:].mean()
    C_AA_ss  = batch.y_points[10, ss_start_idx:].mean()
    print(f"  Batch {i}:")
    print(f"    Glucose:      feed={C_Glc_feed:.2f}, steady-state={C_Glc_ss:.2f} mmol/L "
          f"({C_Glc_ss/C_Glc_feed*100:.1f}% of feed)")
    print(f"    Glutamine:    feed={C_Gln_feed:.2f}, steady-state={C_Gln_ss:.2f} mmol/L "
          f"({C_Gln_ss/C_Gln_feed*100:.1f}% of feed)")
    print(f"    Amino Acids:  feed={C_AA_feed:.2f}, steady-state={C_AA_ss:.2f} mmol/L "
          f"({C_AA_ss/C_AA_feed*100:.1f}% of feed)")

print("\n✓ All chemostat-specific validations completed")


Chemostat-Specific Validations:

1. Steady-State Convergence Analysis (Batch 1):
  Xv (idx 0): mean=3.95e+09, std=1.34e+06, CV=0.03%
    ✓ Reached steady-state
  C_Glc (idx 1): mean=8.49e+00, std=2.44e-03, CV=0.03%
    ✓ Reached steady-state
  C_Gln (idx 2): mean=1.66e+00, std=7.30e-04, CV=0.04%
    ✓ Reached steady-state
  C_Lac (idx 3): mean=1.64e+01, std=3.05e-03, CV=0.02%
    ✓ Reached steady-state
  C_Amm (idx 4): mean=3.93e+00, std=7.89e-04, CV=0.02%
    ✓ Reached steady-state
  C_AA (idx 10): mean=7.98e+00, std=6.85e-04, CV=0.01%
    ✓ Reached steady-state

2. Growth Rate vs Dilution Rate:
  Batch 1: mu_mean=0.0200, D_mean=0.0200 1/h
           mu/D ratio: 1.000
           Net growth (mu-D): 0.0000 1/h
  Batch 2: mu_mean=0.0200, D_mean=0.0200 1/h
           mu/D ratio: 1.000
           Net growth (mu-D): 0.0000 1/h
  Batch 3: mu_mean=0.0200, D_mean=0.0200 1/h
           mu/D ratio: 1.000
           Net growth (mu-D): 0.0000 1/h
  Batch 4: mu_mean=0.0200, D_mean=0.0200 1/h
      

## 8. Visualization


In [7]:
from animakinetica.analysis.batch_visualization import plot_batch_biomass_metabolites
from animakinetica.analysis.process_visualization import plot_simulation_profiles

# Biomass, metabolites, and amino acids — 3-panel plot for batch 1
fig_batch = plot_batch_biomass_metabolites(sim, batch_number=1)
fig_batch.show()


In [8]:
# Multi-batch profile overlay — includes amino acids profile (fig index 5)
fig_dict = sim.data.plot_all(1, 1)

# Show profile
fig_dict["batch_volume_feed"].show()



Generating all plots for Chemostat...

[1/16] Plotting batch 1: Biomass & Metabolites...
[2/16] Plotting batch 1: Volume & Feed...
[3/16] Plotting batch 1: Gas Transfer...
[4/16] Plotting batch 1: Nutrient Factors...
[5/16] Plotting batch 1: Inhibition Factors...
[6/16] Plotting batch 1: Kinetic Rates...
[7/16] Plotting batch 1: Complete Kinetics Overview...
[8/16] Plotting batch 1: Stoichiometric Yields...
[9/16] Plotting cumulative harvest...
[10/16] Plotting cumulative medium use...
[11/16] Plotting cumulative gas use...
[12-16/16] Plotting simulation profiles...

✓ All 16 plots generated successfully!



## Summary

All tests completed successfully for **CHEMOSTAT** operating mode:
- ✓ Simulation setup and execution
- ✓ Single and multi-batch runs
- ✓ Data integrity validation (11 state variables incl. `C_AA` at index 10, + `q_AA` kinetic rate)
- ✓ Performance metrics (`total_DCM_kg`, `STY_kg_m3_day`, `medium_efficiency_kg_m3`, `protein_use_efficiency_kg_kg`, `nutrient_use_efficiency_kg_kg`)
- ✓ Chemostat-specific validations (steady-state convergence incl. `C_AA`, μ/D relationship, substrate levels incl. amino acids)
- ✓ Visualization (3-panel biomass/metabolites/amino acids plot; 12-panel multi-batch profiles incl. amino acids)

**Key Observations:**
- Chemostat mode starts in fed-batch phase until trigger conditions are met (cell density or volume)
- Once triggered, transitions to continuous feed/harvest at constant volume
- Steady-state behavior depends on balance between growth rate (μ) and dilution rate (D)
- At steady-state: μ ≈ D (washout occurs if D > μ_max)
- Amino acid steady-state level reflects balance between `C_AA_feed` supply and cellular uptake (`q_AA`)
